# Validate the Python port of the lab's processing

The lab computes the processed fiber photometry signals of each TDT block with MATLAB code. `hadjas_2025.analysis.processing` ports these steps to Python:

1. the raw photodetector signal carries both LEDs, each modulated at its own frequency;
2. demodulation at each LED frequency gives the 465 nm (calcium) and 405 nm (isosbestic) signals at the raw rate (`Gc_raw`, `af_raw`);
3. both are resampled to 100 Hz (`Gc`, `af`);
4. a bi-exponential fit of the 405 nm signal, scaled to the 465 nm signal by robust regression, gives the baseline;
5. dF/F (`dF`) is 100 * (465 nm signal - baseline) / baseline.

This notebook recomputes the processed signals of one session from its raw signal and compares each step with the lab's signals in the same NWB file, block by block, then summarises the comparison over all sessions. Demodulation and resampling give the lab's signals to floating point precision. The baseline, and so dF/F, differs where the bi-exponential fit differs from MATLAB's; the docstring of the `processing` module explains why.

## Open a session

The NWB file must hold the raw signal and the lab's processed signals, as written by `hadjas-2025 convert <data folder> <output folder>` with the default `processed: lab` of the session config. `recording_type` is the cohort of the subject (a key of `cohorts` in `metadata/session_config.yaml`). Once the files of the current conversion are on the DANDI draft, a session can also be streamed with `load_nwb_from_dandi` (commented out below).

In [ ]:
%matplotlib inline
from pathlib import Path, PureWindowsPath

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO
from scipy import signal

from hnasko_lab_to_nwb.hadjas_2025.analysis.lab_comparison import compare_signals
from hnasko_lab_to_nwb.hadjas_2025.analysis.processed_series import compute_processed_series, get_series_blocks
from hnasko_lab_to_nwb.hadjas_2025.analysis.processing import PROCESSED_RATE, PROCESSED_STREAM_NAMES
from hnasko_lab_to_nwb.hadjas_2025.convert_session import get_led_modulation, load_cohort_metadata

nwbfile_path = Path("H:/hnasko-nwbfiles/sub-C4561_ses-varying_frequencies.nwb")
recording_type = "SN pan GABA recordings"

io = NWBHDF5IO(nwbfile_path, "r")
nwbfile = io.read()
# from hnasko_lab_to_nwb.hadjas_2025.analysis.dandi_streaming import load_nwb_from_dandi
# nwbfile, io = load_nwb_from_dandi("001528", "C4561", "varying-frequencies")
print(nwbfile.session_description)

## Recompute the processed signals

The raw series is in `nwbfile.acquisition`. A session of several TDT blocks is written as one series with timestamps, which is split again at the gaps between blocks; each block is processed separately, as in the lab's code. The modulation frequencies of the two LEDs depend on the recording date (`led_modulation` in the session config). The lab's series of the file are split into the same blocks.

In [ ]:
metadata = load_cohort_metadata(recording_type)
raw_metadata_key = next(
    key
    for key, series_metadata in metadata["FiberPhotometry"].items()
    if key.startswith("Fi1r_") and series_metadata["name"] in nwbfile.acquisition
)
site = raw_metadata_key.removeprefix("Fi1r_")
raw_series = nwbfile.acquisition[metadata["FiberPhotometry"][raw_metadata_key]["name"]]
led_modulation = get_led_modulation(nwbfile.session_start_time)
blocks = compute_processed_series(raw_series=raw_series, led_modulation=led_modulation)
raw_block_bounds, _, _ = get_series_blocks(raw_series)

ophys = nwbfile.processing["ophys"]
lab_blocks = {}
for stream_name in PROCESSED_STREAM_NAMES:
    lab_series = ophys[metadata["FiberPhotometry"][f"{stream_name}_{site}"]["name"]]
    block_bounds, _, _ = get_series_blocks(lab_series)
    data = lab_series.data[:]
    lab_blocks[stream_name] = [data[start:end] for start, end in block_bounds]
    assert len(block_bounds) == len(blocks), f"{lab_series.name} has {len(block_bounds)} blocks, not {len(blocks)}"
print(f"Raw series {raw_series.name}, LED modulation {led_modulation}")
print(f"{len(blocks)} block(s), starting at {[round(float(block['starting_time']), 1) for block in blocks]} s")

For each block and stream, `compare_signals` gives the correlation, the largest absolute difference and the standard deviation of the lab's signal over the samples both signals have. `relative_difference` is the largest difference divided by that standard deviation.

In [ ]:
rows = []
for index, block in enumerate(blocks):
    for stream_name in PROCESSED_STREAM_NAMES:
        comparison = compare_signals(recomputed=block[stream_name], lab=lab_blocks[stream_name][index])
        rows.append(dict(block=index + 1, stream=stream_name, **comparison))
session_comparison = pd.DataFrame(rows)
session_comparison["relative_difference"] = (
    session_comparison["max_absolute_difference"] / session_comparison["lab_standard_deviation"]
)
session_comparison

## Look at one block, step by step

Choose the block (from 0). The plots below show the lab's signal in blue, the recomputed one dashed in orange, and their difference in gray. Times are from the start of the block, with sample k (from 1) at k / rate, as in the lab's code.

In [ ]:
block_index = 0

block = blocks[block_index]
lab = {stream_name: lab_blocks[stream_name][block_index] for stream_name in PROCESSED_STREAM_NAMES}
raw_start, raw_end = raw_block_bounds[block_index]
raw = np.asarray(raw_series.data[raw_start:raw_end])
raw_rate = block["sampling_frequency"]
raw_time = np.arange(1, len(raw) + 1) / raw_rate
processed_time = np.arange(1, len(block["Gc"]) + 1) / PROCESSED_RATE
LAB_COLOR, RECOMPUTED_COLOR, DIFFERENCE_COLOR = "tab:blue", "tab:orange", "0.35"


def plot_step(time, lab_signal, recomputed_signal, title, ylabel, zoom, scale=1.0):
    """Plot the lab's and the recomputed signal over the block and a zoom, and their difference."""
    number_of_samples = min(len(lab_signal), len(recomputed_signal))
    time = time[:number_of_samples]
    lab_signal = scale * np.asarray(lab_signal[:number_of_samples])
    recomputed_signal = scale * np.asarray(recomputed_signal[:number_of_samples])
    difference = recomputed_signal - lab_signal
    step = max(1, number_of_samples // 50000)  # thin the whole-block views of raw-rate signals
    figure, axes = plt.subplots(1, 3, figsize=(16, 3.4), layout="constrained")
    in_zoom = (time >= zoom[0]) & (time < zoom[1])
    for ax, selection, label in [
        (axes[0], slice(None, None, step), "whole block"),
        (axes[1], in_zoom, f"{zoom[0]:g} to {zoom[1]:g} s"),
    ]:
        ax.plot(time[selection], lab_signal[selection], color=LAB_COLOR, lw=2.5, label="lab (MATLAB)")
        ax.plot(
            time[selection],
            recomputed_signal[selection],
            color=RECOMPUTED_COLOR,
            lw=1,
            ls="--",
            label="recomputed (Python)",
        )
        ax.set(title=f"{title}, {label}", xlabel="time in block (s)", ylabel=ylabel)
    axes[0].legend(fontsize=8)
    axes[2].plot(time[::step], difference[::step], color=DIFFERENCE_COLOR, lw=0.6)
    axes[2].set(
        title=f"recomputed minus lab, largest {np.max(np.abs(difference)):.2g}",
        xlabel="time in block (s)",
        ylabel=ylabel,
    )
    return figure

### 1. Raw signal

The raw photodetector signal sums the light of both LEDs, each switched on and off at its own modulation frequency. Its power spectrum should show a peak at each of the two frequencies of the session config (dashed lines); if not, the LED frequencies used for demodulation are wrong for this recording date.

In [ ]:
figure, (ax_trace, ax_spectrum) = plt.subplots(1, 2, figsize=(16, 3.4), layout="constrained")
in_window = (raw_time >= 10) & (raw_time < 10.05)
ax_trace.plot(raw_time[in_window], 1000 * raw[in_window], color=DIFFERENCE_COLOR, lw=1)
ax_trace.set(title="raw signal, 50 ms", xlabel="time in block (s)", ylabel="raw signal (mV)")
frequencies, power = signal.welch(raw[: int(60 * raw_rate)], fs=raw_rate, nperseg=2**15)
ax_spectrum.semilogy(frequencies, power, color=DIFFERENCE_COLOR, lw=1)
for key, label, color in [
    ("calcium_signal_in_hz", "465 nm LED", "tab:green"),
    ("isosbestic_control_in_hz", "405 nm LED", "tab:purple"),
]:
    ax_spectrum.axvline(led_modulation[key], color=color, ls="--", lw=1.5, label=f"{label}, {led_modulation[key]:g} Hz")
ax_spectrum.set(
    xlim=(0, 600), title="power spectrum of the first minute", xlabel="frequency (Hz)", ylabel="power (V²/Hz)"
)
ax_spectrum.legend(fontsize=8);

### 2. Demodulation

The raw signal is multiplied by a sine and a cosine at each LED frequency, both products are lowpass filtered (6 Hz) forward and backward, and the amplitude is the square root of the sum of their squares. The recomputed signals should lie on the lab's, with a difference at floating point level (about 1e-10 mV, that is 1e-13 V).

In [ ]:
plot_step(raw_time, lab["Gc_raw"], block["Gc_raw"], "465 nm demodulated", "465 nm (mV)", zoom=(10, 12), scale=1000)
plot_step(raw_time, lab["af_raw"], block["af_raw"], "405 nm demodulated", "405 nm (mV)", zoom=(10, 12), scale=1000);

### 3. Resampling to 100 Hz

The demodulated signals are resampled to 100 Hz by linear interpolation. Below, the 100 Hz samples (dots: lab, crosses: recomputed) should fall on the raw-rate demodulated signal (line), and the recomputed 100 Hz signals should lie on the lab's.

In [ ]:
figure, ax = plt.subplots(figsize=(16, 3.4), layout="constrained")
in_raw_window = (raw_time >= 10) & (raw_time < 10.2)
in_processed_window = (processed_time >= 10) & (processed_time < 10.2)
ax.plot(
    raw_time[in_raw_window],
    1000 * block["Gc_raw"][in_raw_window],
    color=DIFFERENCE_COLOR,
    lw=1,
    label="demodulated, raw rate",
)
ax.plot(
    processed_time[in_processed_window],
    1000 * lab["Gc"][in_processed_window],
    "o",
    color=LAB_COLOR,
    ms=8,
    label="lab, 100 Hz",
)
ax.plot(
    processed_time[in_processed_window],
    1000 * block["Gc"][in_processed_window],
    "x",
    color=RECOMPUTED_COLOR,
    ms=8,
    mew=2,
    label="recomputed, 100 Hz",
)
ax.set(title="465 nm, 10 to 10.2 s", xlabel="time in block (s)", ylabel="465 nm (mV)")
ax.legend(fontsize=8)

plot_step(processed_time, lab["Gc"], block["Gc"], "465 nm at 100 Hz", "465 nm (mV)", zoom=(10, 12), scale=1000)
plot_step(processed_time, lab["af"], block["af"], "405 nm at 100 Hz", "405 nm (mV)", zoom=(10, 12), scale=1000);

### 4. Baseline

The baseline is a bi-exponential fit of the 405 nm signal, leaving out the first second, scaled to the 465 nm signal by robust linear regression. The NWB file does not store the lab's baseline, but it follows from the lab's own signals: since dF/F = 100 * (465 nm - baseline) / baseline, the lab's baseline is 465 nm / (1 + dF/F / 100). Left, the 405 nm signal and the recomputed bi-exponential fit; middle, the 465 nm signal with the lab's and the recomputed baselines; right, their difference in percent of the lab's baseline. Where the baselines differ, the bi-exponential fits differ: MATLAB refines a start point it computes itself, which is not available outside MATLAB. The lab's own fit is in its .mat files; `hadjas-2025 compare-processing --plot-dir-path` plots it next to the recomputed one.

In [ ]:
lab_baseline = lab["Gc"] / (1 + lab["dF"] / 100)
number_of_samples = min(len(lab_baseline), len(block["lin_fit"]))
baseline_difference = (
    100 * (block["lin_fit"][:number_of_samples] - lab_baseline[:number_of_samples]) / lab_baseline[:number_of_samples]
)

figure, axes = plt.subplots(1, 3, figsize=(16, 3.4), layout="constrained")
axes[0].plot(processed_time, 1000 * block["af"], color="0.75", lw=0.5, label="405 nm signal")
axes[0].plot(processed_time, 1000 * block["af_fit"], color=RECOMPUTED_COLOR, lw=2, ls="--", label="recomputed fit")
axes[0].set(title="bi-exponential fit of the 405 nm signal", xlabel="time in block (s)", ylabel="405 nm (mV)")
axes[1].plot(processed_time, 1000 * block["Gc"], color="0.75", lw=0.5, label="465 nm signal")
axes[1].plot(processed_time[: len(lab_baseline)], 1000 * lab_baseline, color=LAB_COLOR, lw=2.5, label="lab baseline")
axes[1].plot(
    processed_time, 1000 * block["lin_fit"], color=RECOMPUTED_COLOR, lw=1.5, ls="--", label="recomputed baseline"
)
axes[1].set(title="baseline of dF/F", xlabel="time in block (s)", ylabel="465 nm (mV)")
axes[2].plot(processed_time[:number_of_samples], baseline_difference, color=DIFFERENCE_COLOR, lw=1)
axes[2].set(
    title=f"recomputed minus lab baseline, largest {np.max(np.abs(baseline_difference)):.2g} %",
    xlabel="time in block (s)",
    ylabel="% of the lab baseline",
)
for ax in axes[:2]:
    ax.legend(fontsize=8)

### 5. dF/F

dF/F follows from the 465 nm signal and the baseline. Where the baselines agree, dF/F agrees; where they differ, dF/F is offset by the slowly varying difference of the baselines, and the points of the scatter plot (right) leave the diagonal.

In [ ]:
plot_step(processed_time, lab["dF"], block["dF"], "dF/F", "dF/F (%)", zoom=(60, 120))

number_of_samples = min(len(lab["dF"]), len(block["dF"]))
figure, ax = plt.subplots(figsize=(4.5, 4.5), layout="constrained")
limits = np.percentile(lab["dF"], [0.1, 99.9])
ax.plot(limits, limits, color="0.6", lw=1, ls="--", label="recomputed = lab")
ax.plot(
    lab["dF"][:number_of_samples],
    block["dF"][:number_of_samples],
    ".",
    color=RECOMPUTED_COLOR,
    ms=1,
    alpha=0.3,
    label="samples",
)
ax.set(xlim=limits, ylim=limits, xlabel="lab dF/F (%)", ylabel="recomputed dF/F (%)", title="dF/F, every sample")
ax.legend(fontsize=8);

In [ ]:
io.close()

## All sessions

`hadjas-2025 compare-processing <data folder> <output folder>` makes the same comparison for every block of every session in the lab's data share, against the lab's .mat files, and writes `processing_comparison.csv` with one row per block and stream (and, with `--plot-dir-path`, one figure per session with the lab's and the recomputed bi-exponential fits). It needs the data share, so it is run separately; set the path to its output below.

The table counts, per cohort and stream, the blocks whose largest difference is below 1 % of the standard deviation of the lab's signal. The largest difference of dF/F is often at the start or the end of a recording, where the fits differ most. Blocks whose fit the lab adjusted by hand (`LAB_ADJUSTED_FITS` in `lab_comparison`) are left out of the table and listed below, since their dF/F is not expected to match.

In [ ]:
comparison_file_path = Path("H:/Hnasko_dFF_comparison_Python_vs_MATLAB/processing_comparison.csv")

all_comparisons = pd.read_csv(comparison_file_path)
compared = all_comparisons[all_comparisons["error"].isna()].copy()
compared["cohort"] = compared["protocol_folder_path"].map(
    lambda path: next(part for part in PureWindowsPath(path).parts if part.endswith("recordings"))
)
compared["relative_difference"] = compared["max_absolute_difference"] / compared["lab_standard_deviation"]
print(
    f"{compared.groupby(['subject_id', 'protocol_folder_path']).ngroups} sessions, {len(compared) // len(PROCESSED_STREAM_NAMES)} blocks"
)

compared[compared["lab_adjusted_fit"].isna()].groupby(["cohort", "stream"]).agg(
    blocks=("relative_difference", "size"),
    median_correlation=("correlation", "median"),
    blocks_within_1_percent=("relative_difference", lambda difference: int((difference < 0.01).sum())),
)

In [ ]:
compared[compared["lab_adjusted_fit"].notna() & (compared["stream"] == "dF")][
    ["subject_id", "protocol", "block", "segment", "lab_adjusted_fit", "correlation"]
]

Sessions whose comparison failed, for example because the lab's .mat file holds another recording than the data share's blocks:

In [ ]:
all_comparisons[all_comparisons["error"].notna()][["subject_id", "protocol", "error"]]